# 02_Image_Download
Creates Google Earth Engine export tasks from image_metadata.csv.

In [ ]:

!pip -q install earthengine-api geemap pandas tqdm

import ee
import pandas as pd
from tqdm import tqdm
from google.colab import files

ee.Authenticate()
ee.Initialize(project="heatwave-flood-prediction")

print("Upload image_metadata.csv")
uploaded=files.upload()
filename=list(uploaded.keys())[0]
df=pd.read_csv(filename)

EXPORT_FOLDER="FloodProject"
BUFFER_METERS=2500
SENTINEL_SCALE=10
LANDSAT_SCALE=30

for _,row in tqdm(df.iterrows(),total=len(df)):
    event_id=row["event_id"]
    sat=row["satellite"]
    image_date=row["image_date"]
    lat=row["latitude"]
    lon=row["longitude"]

    point=ee.Geometry.Point([lon,lat])
    region=point.buffer(BUFFER_METERS).bounds()

    start=ee.Date(image_date)
    end=start.advance(1,"day")

    if sat=="Sentinel-2":
        image=(ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
               .filterBounds(point)
               .filterDate(start,end)
               .first())
        scale=SENTINEL_SCALE
        folder=EXPORT_FOLDER+"/Sentinel"
    else:
        image=(ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
               .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
               .filterBounds(point)
               .filterDate(start,end)
               .first())
        scale=LANDSAT_SCALE
        folder=EXPORT_FOLDER+"/Landsat"

    task=ee.batch.Export.image.toDrive(
        image=image,
        description=event_id,
        folder=folder,
        fileNamePrefix=event_id,
        region=region,
        scale=scale,
        maxPixels=1e13,
        fileFormat="GeoTIFF"
    )
    task.start()
    print("Started:",event_id)

print("Finished creating export tasks.")
